# Text summarisation with transformer model


NLP (Natural Language Processing) with PEFT (Parameter Efficient Fine-Tuning) and LoRA (Low-Rank Adaptation) for Dialogue Summarization

## Introduction

Summarisation is an important task in natural language processing and useful for a number of cases. For example, to recup a paper, an article or a long email. 

This example leverages pre-trained Language Models, specifically Transformers, a model architecture introduced in the paper "Attention is All You Need". There are many pre-trained available transformers, including ones good for summarisation tasks, basically a Sequence2Sequence network (encoder-decoder).

Pre-trained transformers are already good out-of-the-box but can be further fine tuned on specific smaller datasets, in this case we use a dialogues dataset.
The extra training is done using the efficient Parameter Efficient Fine-Tuning (PEFT).



In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)


# Load the dataset

In [3]:
dataPath = "../datasets/dialog/CSV/"
train = pd.read_csv(dataPath+'train.csv')

In [4]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12460 entries, 0 to 12459
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   id        12460 non-null  object
 1   dialogue  12460 non-null  object
 2   summary   12460 non-null  object
 3   topic     12460 non-null  object
dtypes: object(4)
memory usage: 389.5+ KB


In [5]:
nullColumns = train.columns[train.isnull().any()]

# Print the null columns if any
if len(nullColumns) > 0:
    print("Null columns found:")
    print(nullColumns)
else:
    print("No null columns found.")

No null columns found.


In [6]:
train.isna().sum()


id          0
dialogue    0
summary     0
topic       0
dtype: int64

No invalid value found

In [7]:
train.duplicated().sum()


0

No duplicates

In [8]:
train.head(3)

,id,dialogue,summary,topic
0,train_0,"#Person1#: Hi, Mr. Smith. I'm Doctor Hawkins. ...","Mr. Smith's getting a check-up, and Doctor Haw...",get a check-up
1,train_1,"#Person1#: Hello Mrs. Parker, how have you bee...",Mrs Parker takes Ricky for his vaccines. Dr. P...,vaccines
2,train_2,"#Person1#: Excuse me, did you see a set of key...",#Person1#'s looking for a set of keys and asks...,find keys


For the model we will use only the dialogue and the summary columns.

In [9]:
test = pd.read_csv(dataPath+'test.csv')
val = pd.read_csv(dataPath+'validation.csv')

In [10]:
# Print the sizes of the datasets
print("Train dataset size:", len(train))
print("Test dataset size:", len(test))
print("Validation dataset size:", len(val))

Train dataset size: 12460
Test dataset size: 1500
Validation dataset size: 500


# Load the base transformer model

In [11]:
%pip install evaluate

Note: you may need to restart the kernel to use updated packages.


In [12]:
%pip install rouge_score

Note: you may need to restart the kernel to use updated packages.


In [13]:
from transformers import AutoModelForSeq2SeqLM
from transformers import AutoTokenizer

import torch


Fine-tuning training is very resource-hungry so let's use GPU when possible.

In [14]:
def getDevice():
    """
    Get the best available device with proper MPS handling.
    """
    if torch.cuda.is_available():
        return torch.device("cuda")
    elif torch.backends.mps.is_available():
        # Check if MPS is properly supported
        try:
            # Test MPS with a simple operation
            test_tensor = torch.randn(2, 2, device="mps")
            _ = test_tensor + test_tensor
            return torch.device("mps")
        except Exception as e:
            print(f"MPS available but not working properly: {e}")
            print("Falling back to CPU")
            return torch.device("cpu")
    else:
        return torch.device("cpu")

# Set device
myDevice = getDevice()
print(f"Using device: {myDevice}")

Using device: cuda


Load the pre-trained Google's [FLAN-T5 model](https://huggingface.co/google/flan-t5-base) and its tokeniser directly from Hugging Face. We'll be using the base version of FLAN-T5 for this project.

To optimise memory usage, we set `torch_dtype=torch.bfloat16` to specify the memory type used by this model.  

You can ignore the warning when loading the model.

In [15]:
baseModelName='google/flan-t5-base'
baseModel = AutoModelForSeq2SeqLM.from_pretrained(baseModelName, torch_dtype=torch.bfloat16, device_map="auto")


2025-08-06 07:25:01.419832: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1754465101.446919     190 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1754465101.455328     190 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [16]:
print("Model is on:", next(baseModel.parameters()).device)


Model is on: cuda:0


In [17]:
tokeniser = AutoTokenizer.from_pretrained(baseModelName)

# Inference using the base model

The base model is immediately usable to summarise text.  
Here is how to tokenise a sentence:

In [18]:
# Define a test sentence
sentence = "This is a test sentence."

# Encode the sentence using the tokeniser, returning PyTorch tensors
sentenceEncoded = tokeniser(sentence, return_tensors='pt')

# Decode the encoded sentence, skipping special tokens (end / start)
sentenceDecoded = tokeniser.decode(
        sentenceEncoded["input_ids"][0], 
        skip_special_tokens=True
    )

# Print the encoded sentence's representation
print('ENCODED SENTENCE:')
print(sentenceEncoded["input_ids"][0])

# Print the decoded sentence
print('\nDECODED SENTENCE:')
print(sentenceDecoded)

ENCODED SENTENCE:
tensor([ 100,   19,    3,    9,  794, 7142,    5,    1])

DECODED SENTENCE:
This is a test sentence.


##  Zero Shot Inferencing

Zero-shot inference means that we ask the model directly to summarise a given text, without providing any example. 
This is the most basic prompt and probably the most used questions by users: "here is a text, please give me its summary".

In [19]:
test.head(1)

,id,dialogue,summary,topic
0,test_0_1,"#Person1#: Ms. Dawson, I need you to take a di...",Ms. Dawson helps #Person1# to write a memo to ...,communication method


In [20]:
TEST_IDX = 42  # feel free to try other test sentences

In [21]:
print(test.loc[TEST_IDX].dialogue)

#Person1#: I don't know how to adjust my life. Would you give me a piece of advice?
#Person2#: You look a bit pale, don't you?
#Person1#: Yes, I can't sleep well every night.
#Person2#: You should get plenty of sleep.
#Person1#: I drink a lot of wine.
#Person2#: If I were you, I wouldn't drink too much.
#Person1#: I often feel so tired.
#Person2#: You better do some exercise every morning.
#Person1#: I sometimes find the shadow of death in front of me.
#Person2#: Why do you worry about your future? You're very young, and you'll make great contribution to the world. I hope you take my advice.


This is the prompt that can be used for a zero-shot:

In [22]:
def makeZeroPrompt(testSentence):
    
    dialogue2summarise = test.loc[testSentence].dialogue

    prompt = f"""
Summarise the following conversation.

{dialogue2summarise}

Summary:
    """
    return prompt

In [23]:
prompt = makeZeroPrompt(TEST_IDX)
print(prompt)


Summarise the following conversation.

#Person1#: I don't know how to adjust my life. Would you give me a piece of advice?
#Person2#: You look a bit pale, don't you?
#Person1#: Yes, I can't sleep well every night.
#Person2#: You should get plenty of sleep.
#Person1#: I drink a lot of wine.
#Person2#: If I were you, I wouldn't drink too much.
#Person1#: I often feel so tired.
#Person2#: You better do some exercise every morning.
#Person1#: I sometimes find the shadow of death in front of me.
#Person2#: Why do you worry about your future? You're very young, and you'll make great contribution to the world. I hope you take my advice.

Summary:
    


The summary is created using the base model's method '*generate*' given as input the tokenised prompt:

In [24]:
    # Tokenise the dialogue and convert it to a vector of PyTorch tensors
inputs = tokeniser(prompt, return_tensors='pt')

# Move individual tensors to the same device as the model
inputs = {k: v.to(myDevice) for k, v in inputs.items()}

    # Generate an output using the model, limiting the new tokens to 80
    # This uses the LLM to generate a summary of the dialogue without any further examples
originalModelSummary = tokeniser.decode(
        baseModel.generate(
            inputs["input_ids"], 
            max_new_tokens=80,
        )[0], 
        skip_special_tokens=True
    )

    # retrieve the original summary, to compare
summary = test.loc[TEST_IDX].summary

print("ZERO SHOT INFERENCE ==>\n")
print(f'HUMAN SUMMARY:\n{summary}\n')
print(f'BASE MODEL SUMMARY:\n{originalModelSummary}')


ZERO SHOT INFERENCE ==>

HUMAN SUMMARY:
#Person1# wants to adjust #Person1#'s life and #Person2# suggests #Person1# be positive and stay healthy.

BASE MODEL SUMMARY:
Person1: I'm worried about my future.


Result is not that bad, it gives a hint of what is going on but it's also evident that the model faces challenges in summarising the dialogue when compared to the baseline summary. 

## Few shot prompting
A possibility to improve is to provide one or more examples before asking to summarise. 
We prepare a new function that creates a prompt including examples of dialogues and the corresponding summary.

In [25]:
def makePrompt(shotExamples, target):
    """
    Construct a prompt for one-shot or few-shot inference.

    Parameters
    ----------
    shotExamples : list
        A list containing indices for complete dialogues to be included in the prompt. 
        These dialogues serve as examples for the model to learn from 
        (for one-shot or few-shot inference).
    target : int
        The index for the dialogue that the model is expected to give a summary for.

    Returns
    -------
    str
        A prompt string that is constructed as per the given parameters - 
        full dialogues examples followed by a dialogue that needs to be summarised.
    """
    
    prompt = ""

      # Go through each index in the full examples list
    for ex in shotExamples:
        dialogue = train.loc[ex].dialogue
        summary = train.loc[ex].summary

        # Add each dialogue and its summary to the prompt string, followed by a stop sequence. The stop sequence 
        # '{summary}\n\n\n' is essential for FLAN-T5 model. Other models may have their own different stop sequence.
        prompt += f"""
Dialogue:

{dialogue}

What was going on?
{summary}


"""

        # Now add the dialogue that needs to be summarized by the model
    dialogue2summarise = test.loc[target].dialogue

    # start prompt with this target dialogue
    prompt += f"""
Summarise this dialogue:

{dialogue2summarise}

What was going on?
"""

    # Return the constructed prompt
    return prompt

In [26]:
  # Define indices for full examples to be included in the prompt as a few-shot examples 
oneExampleTest = [len(test)]  # take the last one
  # Define the index for the dialogue that the model is expected to give a summary for
targetTest = 1000

  # Create the prompt for one-shot inference
oneShotPrompt = makePrompt(oneExampleTest, targetTest)

print(oneShotPrompt)



Dialogue:

#Person1#: I see you have bought the latest copy of beauty and fashion. Are there any interesting articles in it?
#Person2#: There's an interesting interview with a top fashion designer about the latest fashions. I enjoyed reading her thought. The which? section is very interesting this month. They tested facial cleaners. T
#Person1#: I like to take the tests that they print in this magazine.
#Person2#: Which tests do you mean?
#Person1#: You know. Tests like how jealous are you? and are you a fashion victim?
#Person2#: Oh, I see. I like to do those tests, too, but I don't take them seriously.
#Person1#: Of course not, but sometimes the results make you think about yourself and what you do. According to the jealousy test, I'm quite a jealous type of person. Perhaps I need to control my jealousy.
#Person2#: You're right. It's sometimes hard to tell which pages are advertisement and which ones are articles.

What was going on?
#Person2# asks about the magazine #Person1# bough

As you can see, the prompt now has an example first, then asks to summarise a target dialogue. 
We use the same way: tokenise the prompt and pass the tokens to the base model. De-tokenise and print the output from the model:

In [27]:
baselineSummary= test.loc[targetTest].summary

  # Tokenise the few-shot prompt and convert it to PyTorch tensors
inputs = tokeniser(oneShotPrompt, return_tensors='pt')
  # Move individual tensors to the same device as the model
inputs = {k: v.to(myDevice) for k, v in inputs.items()}

  # Generate an output using the model, limiting the new tokens to 80
  # This uses the LLM to generate a summary of the dialogue with the few-shot prompt
output = tokeniser.decode(
    baseModel.generate(
        inputs["input_ids"],
        max_new_tokens=80,
    )[0], 
    skip_special_tokens=True
)

  # Show the results
print(f'BASELINE SUMMARY:\n{baselineSummary}\n')
print(f'MODEL GENERATION - ONE SHOT:\n{output}')

BASELINE SUMMARY:
#Person1# and #Person2# talk about the difficulty of not having a personal computer.

MODEL GENERATION - ONE SHOT:
Person1 is frustrated because they can't access the computers in the library.


Better! You can even provide more examples:

In [28]:
# Define indices for full examples to be included in the prompt as a few-shot examples 
fewExamplesTest = [0, 100]
# Define the index for the dialogue that the model is expected to give a summary for
targetTest = 1080

# Create the prompt for one-shot inference
fewShotPrompt = makePrompt(fewExamplesTest, targetTest)

print(fewShotPrompt)


Dialogue:

#Person1#: Hi, Mr. Smith. I'm Doctor Hawkins. Why are you here today?
#Person2#: I found it would be a good idea to get a check-up.
#Person1#: Yes, well, you haven't had one for 5 years. You should have one every year.
#Person2#: I know. I figure as long as there is nothing wrong, why go see the doctor?
#Person1#: Well, the best way to avoid serious illnesses is to find out about them early. So try to come at least once a year for your own good.
#Person2#: Ok.
#Person1#: Let me see here. Your eyes and ears look fine. Take a deep breath, please. Do you smoke, Mr. Smith?
#Person2#: Yes.
#Person1#: Smoking is the leading cause of lung cancer and heart disease, you know. You really should quit.
#Person2#: I've tried hundreds of times, but I just can't seem to kick the habit.
#Person1#: Well, we have classes and some medications that might help. I'll give you more information before you leave.
#Person2#: Ok, thanks doctor.

What was going on?
Mr. Smith's getting a check-up, and 

In [29]:
baselineSummary= test.loc[targetTest].summary

  # Tokenise the few-shot prompt and convert it to PyTorch tensors
inputs = tokeniser(fewShotPrompt, return_tensors='pt')
  # Move individual tensors to the same device as the model
inputs = {k: v.to(myDevice) for k, v in inputs.items()}

  # Generate an output using the model, limiting the new tokens to 80
  # This uses the LLM to generate a summary of the dialogue with the few-shot prompt
output = tokeniser.decode(
    baseModel.generate(
        inputs["input_ids"],
        max_new_tokens=80,
    )[0], 
    skip_special_tokens=True
)

  # Show the results
print(f'BASELINE SUMMARY:\n{baselineSummary}\n')
print(f'MODEL GENERATION - ONE SHOT:\n{output}')

Token indices sequence length is longer than the specified maximum sequence length for this model (805 > 512). Running this sequence through the model will result in indexing errors


BASELINE SUMMARY:
#Person1# and #Person2# both enjoy watching the baseball game in a great place with great fans.

MODEL GENERATION - ONE SHOT:
The game between Baltimore and Baltimore is a great one.


You can see from the warning that this model has a low limit for the input: 512 tokens.
The model’s position embedding table is fixed at 512 tokens. Cannot be changed. 
If your use case demands longer context, you can switch to a model designed for that, but they will require mugh higher virtual memory and time.  
For this reason, we will not use the few shots prompt.

You can see that most of the test dialogues are longer than 512 characters:

In [30]:
(test['dialogue'].apply(len)>512).sum()


1038

# Evaluation

It's not straightforward to compare two summaries, i.e. two strings. Different words can convey the same meaning.  
**ROUGE** stands for **Recall-Oriented Understudy for Gisting Evaluation**. It’s a set of metrics used to evaluate automatic summaries or generated texts by comparing them to human-written reference texts.



In [31]:
import evaluate

In [32]:
rouge = evaluate.load('rouge')

Let's see how the last generated summary compares with the human-prepared baseline:

In [33]:
rouge.compute(
    predictions = [output],
    references = [baselineSummary],
    use_aggregator=True,
    use_stemmer=True,
)

{'rouge1': 0.38461538461538464,
 'rouge2': 0.08333333333333334,
 'rougeL': 0.3076923076923077,
 'rougeLsum': 0.3076923076923077}

## ROUGE Metrics Explained

What does ROUGE measure?
It measures the overlap between two texts — usually between a machine-generated summary and a human (gold-standard) summary — using:

ROUGE-N: n-gram overlap (e.g., ROUGE-1 for unigrams, ROUGE-2 for bigrams)

ROUGE-L: longest common subsequence


rouge1 (0.38)

What it measures: Overlap of unigrams (single words) between generated and reference text
Formula: (Number of overlapping words) / (Total words in reference)
Interpretation: 38% of words in the reference text appear in the generated text
Good for: Basic vocabulary coverage

rouge2 (0.08)

What it measures: Overlap of bigrams (2-word sequences) between generated and reference text
Formula: (Number of overlapping bigrams) / (Total bigrams in reference)
Interpretation: 8% of word pairs in the reference appear in the generated text
Good for: Measuring fluency and word order preservation

rougeL (0.31)

What it measures: Longest Common Subsequence (LCS) between generated and reference text
Formula: LCS_length / reference_length
Interpretation: 31% represents the longest sequence of words that appear in the same order
Good for: Measuring structural similarity while allowing gaps

rougeLsum (0.31)

What it measures: ROUGE-L applied to summary-level evaluation
Difference from rougeL: Computed at sentence level then averaged (for multi-sentence summaries)
Interpretation: Same as rougeL in this case
Good for: Multi-sentence summary evaluation

Important Notes

These are recall-based metrics - they measure how much of the reference text is captured
Higher isn't always better - perfect scores might indicate copying rather than good summarization
rouge2 is typically lower than rouge1 since exact phrase matches are harder
Context matters - good scores vary by task (summarization vs. translation vs. generation)

The scores indicate the generated text captures a reasonable amount of the reference content but could improve in maintaining exact phrase structures and word sequences.

## Evaluate base model on test (sub-)dataset
We can evaluate on the Testing dataset (not all, because ROUGE is resource consuming, let's say one every 50).
I will use the zero-shot prompt, as we want to avoid the prompt to be truncated and because the normal use case is just to ask the model to summarise a text, without providing examples.

In [34]:
rouge_scores = []
print("Evaluation started")

for t in range(1,len(test), 50):


      # Create the prompt for zero-shot inference
    zeroShotPrompt = makeZeroPrompt(t)

      # Tokenise the prompt and convert it to PyTorch tensors
    inputs = tokeniser(zeroShotPrompt, return_tensors='pt')
    inputs = inputs.to(myDevice)

        # Generate an output using the model, limiting the new tokens to 80
    output = tokeniser.decode(
        baseModel.generate(
        inputs["input_ids"],
        max_new_tokens=80,
        )[0], 
        skip_special_tokens=True
    )

      # evaluate using ROUGE and store the result
    rouge_scores.append(rouge.compute(
            predictions=[output],
            references=[test.loc[t].summary],
            use_aggregator=True,
            use_stemmer=True
        ))

  # all done
print (f"test sentences evaluated: {len(rouge_scores)} ")

Evaluation started
test sentences evaluated: 30 


For convenience, we convert the results into a dataframe and print the average:

In [35]:
rsBase = pd.DataFrame(rouge_scores)
rsBase.head(3)

,rouge1,rouge2,rougeL,rougeLsum
0,0.260870,0.045455,0.173913,0.173913
1,0.428571,0.000000,0.285714,0.285714
2,0.057143,0.000000,0.057143,0.057143


In [36]:
rsBase.mean().to_dict()


{'rouge1': 0.2239559859191769,
 'rouge2': 0.05870599637005492,
 'rougeL': 0.1930283615857042,
 'rougeLsum': 0.1930283615857042}

**Interpretation Guidelines**
Score Ranges:

0.0-0.2: Poor overlap  
0.2-0.4: Moderate overlap  
0.4-0.6: Good overlap  
0.6+: Excellent overlap  
  
This scores suggest:  
  
Moderate vocabulary overlap (rouge1: 0.22)  
Lower phrase-level similarity (rouge2: 0.06)  
Lower (but almost reasonable) structural alignment (rougeL: 0.19)

# Fine tune the model

Fine-tuning means taking a pre-trained model (like FLAN-T5 or BART) and training it on our specific task (e.g., summarising dialogues), using our custom dataset.
This adjusts the model’s weights to perform better on our use case.

How It Works for Summarisation
We provide input texts (e.g., the conversations in the training dataset) and the target summaries.

The model learns - as ususal in neural networks - to map inputs to outputs by minimising the difference (loss) between its predictions and the target summaries.

We evaluate performance using metrics like ROUGE.

## Prepare the datasets
We transform the existing training, testing and validation datasets by tokenising them,  split into batches and removing unnecessary columns. 
Note that the training test is very large and the training phase will take long time, so the dataset will be reduced.  

The prompt and the labels (target) for the training are also prepared.

In [37]:
  # Helper function
def tokeniseFn(example):
    start_prompt = 'Summarise the following conversation.\n\n'
    end_prompt = '\n\nSummary: '
    prompt = [start_prompt + dialogue + end_prompt for dialogue in example["dialogue"]]
    example['input_ids'] = tokeniser(prompt, padding="max_length", truncation=True, return_tensors="pt").input_ids
    example['labels'] = tokeniser(example["summary"], padding="max_length", truncation=True, return_tensors="pt").input_ids
    
    return example

In [38]:
from datasets import Dataset, load_dataset


In [39]:
  # Convert to Dataset
trainDataset = Dataset.from_pandas(train)
  # tokenise and prepare prompt using map and helper function
tokedTrain = trainDataset.map(tokeniseFn, 
                              batched=True, 
                              remove_columns = trainDataset.column_names)

Map:   0%|          | 0/12460 [00:00<?, ? examples/s]

In [40]:
valDataset = Dataset.from_pandas(val)
tokedVal = valDataset.map(tokeniseFn, 
                          batched=True, 
                          remove_columns = valDataset.column_names)


Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [41]:
testDataset = Dataset.from_pandas(test)
tokedTest = testDataset.map(tokeniseFn, 
                             batched=True, 
                             remove_columns = testDataset.column_names)


Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

Reduce the dataset by filtering them.

In [42]:
  # reduce 20x
tokedTrain = tokedTrain.filter(lambda example, 
                                index: index % 20 == 0, 
                               with_indices=True)

Filter:   0%|          | 0/12460 [00:00<?, ? examples/s]

In [43]:
  # reduce 5x
tokedVal = tokedVal.filter(lambda example, 
                            index: index % 5 == 0, 
                           with_indices=True)

Filter:   0%|          | 0/500 [00:00<?, ? examples/s]

Check the shapes of all three parts of the dataset:

In [44]:
print(f"Shapes of the datasets:")
print(f"Training: {tokedTrain.shape}")
print(f"Validation: {tokedVal.shape}")
print(f"Test: {tokedTest.shape}")

Shapes of the datasets:
Training: (623, 2)
Validation: (100, 2)
Test: (1500, 2)


## PEFT/LoRa for Fine-tuning

PEFT (Parameter-Efficient Fine-Tuning) offers a more efficient alternative to full fine-tuning; it is a set of techniques to fine-tune only a small part of the model, not the entire thing.

Benefits:

- Uses less memory
- Is faster
- Works well even on laptops or limited GPUs

One of the PEFT techniques is Low-Rank Adaptation (LoRA).
Instead of updating full weight matrices, it inserts tiny trainable layers (low-rank matrices) into the model.

The original model weights are frozen, and only the LoRA layers are trained.

Result:

- We get a compact, task-specific model on top of a frozen base model.
- Perfect for tasks like summarisation when resources are limited.

LoRA, in essence, enables fine-tuning of our model with significantly fewer computational resources, sometimes even just a single GPU. 

In [45]:
from peft import get_peft_model, LoraConfig, TaskType, PeftModel, PeftConfig


In [46]:
from transformers import TrainingArguments, Trainer, GenerationConfig
import time

In [47]:
  # use this just to restart training
baseModel = AutoModelForSeq2SeqLM.from_pretrained(baseModelName, 
                                                  device_map="auto")


In [48]:
    # LoraConfig allows to control how LoRA is applied to the base model
LoRaConfiguration = LoraConfig(
    r=32, # rank (int) of the update matrices. Lower rank results in smaller update matrices with fewer trainable parameters
    lora_alpha=32,  #  LoRA scaling factor
    target_modules=["q", "v"],  # The modules to apply the LoRA update matrices: queries, keys, values
    lora_dropout=0.05,
    bias="none",  # no bias parameters should be trained
    task_type=TaskType.SEQ_2_SEQ_LM # FLAN-T5
)

Incorporate LoRA adapter layers and parameters into the original Language Model (LLM) for training.

In [49]:
fineTunedModel = get_peft_model(baseModel, LoRaConfiguration)

In [50]:
fineTunedModel.print_trainable_parameters()


trainable params: 3,538,944 || all params: 251,116,800 || trainable%: 1.4093


### Train the PEFT/LoRA Adapter

In [51]:
outputDir = f'./peft-dialogue-summary-training'

In [52]:

PEFTtrainingArgs = TrainingArguments(
    output_dir = outputDir, # The output directory where the model predictions and checkpoints will be written
    auto_find_batch_size=True,  # fit into memory automatically through exponential decay, avoiding CUDA Out-of-Memory errors
    report_to="none",  # Set to "wandb" or "tensorboard" if using logging tools
    label_names=['labels'],  # targets
    learning_rate=1e-4, # The initial learning rate - Higher learning rate, default is 5e-5
    num_train_epochs=8, # Total number of training epochs to perform (default: 3)
    logging_steps=50,  # Number of update steps between two logs 
    eval_strategy="steps",
    eval_steps=50,  # Evaluate every 20 steps
    save_strategy="steps",
    save_steps=100,      # Number of update steps between two checkpoints
    #max_steps=200,    # the total number of training steps to perform. Overrides num_train_epochs.
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss"
)
    


In [53]:
PEFTtrainer = Trainer(
    model = fineTunedModel,
    args = PEFTtrainingArgs,
    train_dataset = tokedTrain,
    eval_dataset=tokedVal
)

In [54]:
device = next(fineTunedModel.parameters()).device
print(f"Model is on device: {device}")

Model is on device: cuda:0


In [55]:
PEFTtrainer.train()



Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Step,Training Loss,Validation Loss
50,37.971000,27.182579
100,13.480400,5.025587
150,4.178800,3.128888
200,2.875900,1.027745
250,1.510500,0.536661
300,0.911000,0.382443
350,0.671000,0.277540
400,0.520000,0.232624
450,0.442200,0.210622
500,0.395400,0.190803


TrainOutput(global_step=1248, training_loss=2.6910579739472804, metrics={'train_runtime': 802.9651, 'train_samples_per_second': 6.207, 'train_steps_per_second': 1.554, 'total_flos': 3467014878265344.0, 'train_loss': 2.6910579739472804, 'epoch': 8.0})

Loss on validation dataset is decreasing but slowing down, while on the training dataset starts to show some instability.  
You can try for more epochs.  
Now we have a fine tuned model that we can use directly for inference.  
Let's try it with the same example:

In [56]:
TEST_IDX = 42
dialogue = test.loc[TEST_IDX].dialogue
baselineHumanSummary = test.loc[TEST_IDX].summary


prompt = f"""
Summarise the following conversation.

{dialogue}

Summary: """

   # Tokenize the dialogue and convert it to a vector of PyTorch tensors (returns a dictionary)
inputs = tokeniser(prompt, return_tensors='pt', truncation=True, max_length=512)

  # Move tensors to GPU
inputs = {k: v.to(myDevice) for k, v in inputs.items()}

  # Ensure models are on GPU
baseModel = baseModel.to(myDevice)
fineTunedModel = fineTunedModel.to(myDevice)

  #  Generate from PEFT model
peftModelOutputs = fineTunedModel.generate(
    input_ids = inputs["input_ids"],
    attention_mask = inputs["attention_mask"],
    generation_config = GenerationConfig(max_new_tokens=100, num_beams=1)
)
peftModelSummary = tokeniser.decode(peftModelOutputs[0], 
                                          skip_special_tokens=True)


print("ZERO SHOT")
print(f'HUMAN SUMMARY:\n{baselineHumanSummary}')
print("===")
print(f'BASE MODEL SUMMARY:\n{originalModelSummary}')
print("===")
print(f'FINE TUNED MODEL:\n{peftModelSummary}')

ZERO SHOT
HUMAN SUMMARY:
#Person1# wants to adjust #Person1#'s life and #Person2# suggests #Person1# be positive and stay healthy.
===
BASE MODEL SUMMARY:
Person1: I'm worried about my future.
===
FINE TUNED MODEL:
#Person1# wants to adjust his life. #Person2# advises #Person1# to get plenty of sleep. #Person2# advises #Person1# to do some exercise every morning.


The fine tuned model added more details to the summary and is better.

## Evaluate the Model Quantitatively (ROUGE Metric)

The [ROUGE metric](https://en.wikipedia.org/wiki/ROUGE_(metric)) is a valuable tool for assessing the quality of summaries generated by models. It evaluates these summaries by comparing them to a "baseline" summary, typically crafted by a human. Although not flawless, the ROUGE metric provides insights into the improvement in the overall effectiveness of summarization achieved through fine-tuning.

In [57]:
# Define indices for full examples to be included in the prompt as a few-shot examples 
rouge_scores = []
print("Evaluation started")

for t in range(1,len(test), 50):


      # Create the prompt for zero-shot inference
    zeroShotPrompt = makeZeroPrompt(t)

      # Tokenise the prompt and convert it to PyTorch tensors
    inputs = tokeniser(zeroShotPrompt, return_tensors='pt')
    inputs = inputs.to(myDevice)

        # Generate an output using the model, limiting the new tokens to 80
    output = tokeniser.decode(
        baseModel.generate(
        inputs["input_ids"],
        max_new_tokens=80,
        )[0], 
        skip_special_tokens=True
    )

    rouge_scores.append(rouge.compute(
            predictions=[output],
            references=[test.loc[t].summary],
            use_aggregator=True,
            use_stemmer=True
        ))

  # all done
print (f"Test sentences evaluated: {len(rouge_scores)} ")

Evaluation started
Test sentences evaluated: 30 


In [58]:
rsPeft = pd.DataFrame(rouge_scores)

In [59]:
rsPeft.mean().to_dict()


{'rouge1': 0.3187917492785725,
 'rouge2': 0.12031891986607879,
 'rougeL': 0.2667695178107031,
 'rougeLsum': 0.2667695178107031}

In [60]:
rsBase.mean().to_dict()


{'rouge1': 0.2239559859191769,
 'rouge2': 0.05870599637005492,
 'rougeL': 0.1930283615857042,
 'rougeLsum': 0.1930283615857042}

**Score results**

Well, well, after fine tuning the scores improved:  
  
Moderate (almost good) vocabulary overlap (rouge1: 0.32 from 0.22)  
Lower phrase-level similarity (rouge2: 0.12 from 0.06)  
Reasonable structural alignment (rougeL: 0.27 from 0.19)

# Extra: Save the fine tuned model 

In [61]:
peftModelPath="./peft-dialogue-summary-checkpoint-local"


In [62]:

PEFTtrainer.model.save_pretrained(peftModelPath)
tokeniser.save_pretrained(peftModelPath)

('./peft-dialogue-summary-checkpoint-local/tokenizer_config.json',
 './peft-dialogue-summary-checkpoint-local/special_tokens_map.json',
 './peft-dialogue-summary-checkpoint-local/spiece.model',
 './peft-dialogue-summary-checkpoint-local/added_tokens.json',
 './peft-dialogue-summary-checkpoint-local/tokenizer.json')

To use later the model, you need to load the base model first

In [63]:
baseModel = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base", 
                                                  torch_dtype=torch.bfloat16)


Get it from the saved one

In [64]:
tokeniser = AutoTokenizer.from_pretrained(peftModelPath)


In [65]:

# Step 3: Load the PEFT (LoRA) adapter weights
peftModel = PeftModel.from_pretrained(baseModel, peftModelPath)


In [71]:
TEST_IDX = 500
dialogue = test.loc[TEST_IDX].dialogue
baselineHumanSummary = test.loc[TEST_IDX].summary


prompt = f"""
Summarize the following conversation.

{dialogue}

Summary: """

   # Tokenize the dialogue and convert it to a vector of PyTorch tensors (returns a dictionary)
inputs = tokeniser(prompt, return_tensors='pt', 
                   truncation=True, 
                   max_length=512)

# Move tensors to GPU
inputs = {k: v.to(myDevice) for k, v in inputs.items()}

# Ensure models are on GPU
peftModel = peftModel.to(myDevice)

#  Generate from PEFT model
peftModelOutputs = peftModel.generate(
    input_ids=inputs["input_ids"],
    attention_mask=inputs["attention_mask"],
    generation_config=GenerationConfig(max_new_tokens=100, num_beams=1)
)
peftModelSummary = tokeniser.decode(peftModelOutputs[0], 
                                          skip_special_tokens=True)


print("ZERO SHOT")
print(f'HUMAN SUMMARY:\n{baselineHumanSummary}')
print("===")
print(f'FINE TUNED MODEL:\n{peftModelSummary}')

ZERO SHOT
HUMAN SUMMARY:
#Person2# tells David the plan for a tour and #Person2# will celebrate #Person2#'s brother's fortieth birthday when at Salt Lake City.
===
FINE TUNED MODEL:
#Person2# and #Person2# are planning a four day drive to Salt Lake City.
